# [Deprecated] This file has been deprecated.

### Setup

In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42
torch set to use deterministic algorithms


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5,
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[bhiravi_vaidhy, dilip_satgare, haresh_mehta, ...",USA,siddharth_randeria,Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,mel_gibson,Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[dylan_walsh, laura_linney, ernie_hudson_jr, t...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.75,
    val_ratio=0.1,
    test_ratio=0.15,
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 358027 (74.84%)
valid: 46916 (9.81%)
test: 73461 (15.36%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.555944
1    0.444056
Name: proportion, dtype: float64
valid label
0    0.610772
1    0.389228
Name: proportion, dtype: float64
test label
0    0.585277
1    0.414723
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [7]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 358027
  Num of positive interactions: 158984 

Building edges...
Building labels...
Interaction Graph: Data(edge_index=[2, 317967], edge_label=[158984])
Edge Index: tensor([[    0,     0,     0,  ..., 10758, 10762, 10763],
        [ 2089,  2202,  2318,  ...,  1760,  1645,  1760]])


#### Evaluate User Diversity Preference Scale

In [8]:
user_dps_df = evaluator.eval_user_diversity_preference_scale(encoded_train_df, feature_engineer.vocab2idx, normalized=True)
user_dps_df.head(1)

Calculating user diversity preference scale:   0%|          | 0/2064 [00:00<?, ?it/s]

Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:28<00:00, 72.88it/s] 


,userID,actorID_wvec,actorID_dps,country_wvec,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps
0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.511464,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.159621,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.424931,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.755545


#### Prepare train/valid triplet data

In [9]:
train_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_train_df, k_negative_samples=5)
train_triplet_with_dps_df = train_triplet_df.merge(user_dps_df, on="userID", how="left")
train_triplet_with_dps_df.head(1)

Original data count (positive samples): 158984
Num of triplets: 158984(pos samples) * 5(negative sampled items) = 794920


,userID,pos_item_id,neg_item_id,actorID_idx,country_idx,directorID_idx,genre_idx,neg_actorID_idx,neg_country_idx,neg_directorID_idx,neg_genre_idx,actorID_wvec,actorID_dps,country_wvec,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps
0,0,1102,307,"[16600, 10247, 4811, 7720, 13670]",63,2528,"[1, 2, 16, 17, 0, 0, 0, 0]","[13135, 5650, 7287, 9910, 16230]",24,2946,"[8, 10, 0, 0, 0, 0, 0, 0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.511464,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.159621,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.424931,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.755545


#### Prepare prediction pool for inference/testing

In [10]:
# NOTE: Prepare prediction pool to evaluate the model
valid_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_valid_df, K=100)
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, K=500)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2063
Item Pool: 6098, negative sampled to 100 items for each user
Num of interactions: 2063(users) * 100(items) = 206300
Prediction DataFrame:
User Pool: 2064
Item Pool: 6959, negative sampled to 500 items for each user
Num of interactions: 2064(users) * 500(items) = 1032000


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
1031995,2063,447,0,"[15727, 2894, 10573, 253, 3728]",62,2985,"[8, 0, 0, 0, 0, 0, 0, 0]"
1031996,2063,3601,0,"[11716, 4113, 7140, 7336, 6954]",63,2854,"[1, 18, 0, 0, 0, 0, 0, 0]"
1031997,2063,7982,0,"[6627, 283, 3901, 11769, 14462]",63,1419,"[11, 0, 0, 0, 0, 0, 0, 0]"
1031998,2063,5969,0,"[11666, 10099, 4687, 6007, 6501]",63,3441,"[8, 15, 0, 0, 0, 0, 0, 0]"
1031999,2063,5025,0,"[3962, 7760, 5111, 6372, 4643]",62,558,"[6, 11, 14, 17, 0, 0, 0, 0]"


### Prepare DataLoader

In [11]:
# NOTE: ensure reproducibility of DataLoader
import torch
from common.utils import seed_worker
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

# TODO: determine which Dataset to use
from torch.utils.data import DataLoader
from common.datasets import TripletDataset, UserItemPairDataset

BATCH_SIZE = 1024

train_dataset = TripletDataset(train_triplet_with_dps_df)
valid_dataset = UserItemPairDataset(valid_pool_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker, generator=g, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 794920
valid data count: 206300
test data count: 1032000


### Configure Model (LightningModule)

In [ ]:
from lightning_models.mtdp_ngcf_s import MTDPRecV1

EMB_DIM = 32
LR = 1e-3
EPOCHS = 50
NUM_LAYERS = 3
REG_WEIGHT = 1e-5

DPS_WEIGHTS = {
    "actor_dps": 0.25,
    "country_dps": 0.25,
    "director_dps": 0.25,
    "genre_dps": 0.25,
}

MT_WEIGHTS = {
    "rec_loss": 1.0,
    "dps_loss": 0.5,
}

model = MTDPRecV1(
    graph_data=train_graph,  # shape [2, num_edges]
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    embedding_dim=EMB_DIM,
    num_layers=NUM_LAYERS,
    node_dropout=0.0,
    mess_dropout=0.1,
    lr=LR,
    reg_weight=REG_WEIGHT,
    dps_weights=DPS_WEIGHTS,
    mt_weights=MT_WEIGHTS,
)


Seed set to 42


### Configure Trainer and Experiment

In [ ]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "mtdp-s-exp"
RUN_NAME = "run1"
PATIENCE = 5
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME)
trainer_callbacks = get_callbacks(
    exp_name=EXPERIMENT_NAME,
    run_name=RUN_NAME,
    patience=PATIENCE,
    monitor_metric="val_ndcg10",
    monitor_mode="max",
    hyper_param_str=f"n_user={TRAIN_NUM_USERS}-n_item={TRAIN_NUM_ITEMS}-emb_dim={EMB_DIM}-num_layers={NUM_LAYERS}-lr={LR}-reg_weight={REG_WEIGHT}",
)

In [21]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    callbacks=trainer_callbacks,
    accelerator='auto',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Train Model

In [22]:
# Start training
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/adam/R11_Bai/DPRecSys/ablation_experiments/test_checkpoints/mtdp-v1-exp exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name        | Type         | Params | Mode 
-----------------------------------------------------
0 | ngcf_model  | NGCF         | 351 K  | train
1 | bpr_loss    | BPRLoss      | 0      | train
2 | reg_loss    | EmbLoss      | 0      | train
3 | dps_module  | DPSPredictor | 516    | train
4 | dps_loss_fn | DPSLoss      | 0      | train
-----------------------------------------------------
351 K     Trainable params
0         Non-trainable params
351 K     Total params
1.406     Total estimated model params size (MB)
29        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 4, global step 3885: 'val_ndcg10' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_ndcg10 did not improve in the last 5 records. Best score: 0.494. Signaling Trainer to stop.
Epoch 5, global step 4662: 'val_ndcg10' was not in top 1


🏃 View run run1 at: http://140.112.106.216:3683/#/experiments/7/runs/3d9d6d0353ac48af996e0c2ff0faa89e
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/7


### Inference

In [ ]:
# NOTE: the inference model MUST be the same as the training model
best_model_experiment_name = "mtdp-s-exp"
best_model_checkpoint_path = "run0-n_user=2064-n_item=8706-emb_dim=32-num_layers=3-lr=0.001-reg_weight=1e-05-best-checkpoint-epoch=00-val_ndcg10=0.51.ckpt"
best_model_path = f"test_checkpoints/{best_model_experiment_name}/{best_model_checkpoint_path}"

model = MTDPRecV1.load_from_checkpoint(checkpoint_path=best_model_path)
# start inference
trainer.test(model=model, dataloaders=test_loader)


Seed set to 42
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        test_ndcg10        │    0.3688335120677948     │
│        test_ndcg20        │    0.39760616421699524    │
│        test_ndcg5         │    0.32316097617149353    │
│     test_precision10      │    0.13120155036449432    │
│     test_precision20      │    0.1120397299528122     │
│      test_precision5      │    0.14641472697257996    │
│       test_recall10       │    0.11643126606941223    │
│       test_recall20       │    0.18988777697086334    │
│       test_recall5        │    0.06848163902759552    │
└───────────────────────────┴───────────────────────────┘

🏃 View run run0 at: http://140.112.106.216:3683/#/experiments/7/runs/65f9c40bcb7a4c76af3d70a1f2dd3b06
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/7


[{'test_ndcg5': 0.32316097617149353,
  'test_ndcg10': 0.3688335120677948,
  'test_ndcg20': 0.39760616421699524,
  'test_precision5': 0.14641472697257996,
  'test_precision10': 0.13120155036449432,
  'test_precision20': 0.1120397299528122,
  'test_recall5': 0.06848163902759552,
  'test_recall10': 0.11643126606941223,
  'test_recall20': 0.18988777697086334}]

In [18]:
model.test_results["eval_score_df"].describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.323161,0.068482,0.146415,0.368834,0.116431,0.131202,0.397606,0.189888,0.112040
std,595.969798,0.369114,0.121064,0.186757,0.328784,0.152453,0.140615,0.284612,0.195075,0.107653
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,515.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.231378,0.035714,0.050000
50%,1031.500000,0.000000,0.000000,0.000000,0.386853,0.066667,0.100000,0.408231,0.142857,0.100000
75%,1547.250000,0.630930,0.090909,0.200000,0.605260,0.166667,0.200000,0.585390,0.281250,0.150000
max,2063.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.900000,1.000000,1.000000,0.800000
